# Omnibus — Flex-bus eligibility & recommendation

**Goal of this notebook:** decide *which buses are flex buses* (a published designation riders/operators are aware of), then watch the `recommend()` selector pick which one to redeploy for an event — and iterate on both rules.

The fixed GTFS schedule stays untouched. The flex fleet is built on **one idea — thin a high-frequency trunk (Model B)**:

| Model | Idea | Example |
|---|---|---|
| **B — thin a high-frequency trunk** ✅ *shipped designation* | A corridor where **a lot of buses already run** can drop one and still hold a tight headway. Frequency/redundancy is the signal — not low demand. | Line **1**: ~5×/h (~11-min headway) → keep 3×/h (20-min) frees **~2 buses/h**, riders barely notice. |
| **A — divert an underused peripheral line** ❌ *rejected* | Tempting to pull a near-empty line like 77, but that line's *one* bus **is** the service — pulling it strands the route. Low demand is the wrong signal. | kept below only as the foil; **not** how the fleet is chosen. |

It reduces to one question per `(line, direction, hour)`: **can we take a bus without breaking the timetable?** — i.e. is `trips_per_hour` high enough that, after thinning to the headway floor, `buses_freed > 0`.

Inputs:
- `route_efficiency.parquet` — GTFS `trips_per_hour` × dwell-`demand` per (line, dir, daytype, hour), now carrying `headway_min` + `buses_freed`.
- `flex_candidates.parquet` — the flex designation (`flex_eligible`), now Model-B: trunks that can spare a bus for ≥ `MIN_DONOR_HOURS` hours.

> ⚠️ Demand here is the **normalized, baseline** dwell proxy (typical Oct term-time, *not* event-aware). Units are relative — compare lines to each other, don't read absolute riders. In Model B demand is only a sanity check that the trunk is real; the event spike comes from the separate demand-prediction model. This notebook finds *where the spare frequency is on a normal day*.

In [ ]:
import sys, json, math
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

P = "../data/parquet"
eff   = pl.read_parquet(f"{P}/route_efficiency.parquet")
flex  = pl.read_parquet(f"{P}/flex_candidates.parquet")
lines = pl.read_parquet(f"{P}/lines.parquet")
print("efficiency grid:", eff.shape, "| flex fleet:", flex.shape, "| lines:", lines.shape)
eff.head()

## Parameters — tweak these and re-run the notebook

Everything below reacts to these. `HEADWAY_MAX_MIN` is the worst headway we'll accept after thinning a trunk — it's the only knob that drives the **shipped** Model-B designation (a line is a donor in an hour when `trips_per_hour` exceeds `ceil(60/HEADWAY_MAX_MIN)`). `DEMAND_PCTL` / `MIN_TRIPS` only feed the **rejected** Model-A foil further down; they no longer decide the fleet.

In [ ]:
DAYTYPE         = "weekday"   # weekday | sat | sun
HOUR            = 15          # the moment to inspect (e.g. football kickoff @ 15:00)
HEADWAY_MAX_MIN = 20          # Model B (shipped): max acceptable headway after thinning
MIN_DONOR_HOURS = 3           # Model B: must be able to spare a bus this many hours to be published
DEMAND_PCTL     = 0.25        # Model A foil only: "over-served" = load-per-bus bottom quantile
MIN_TRIPS       = 2           # Model A foil only: never pull a line below this many buses/hour

KEEP_TRIPS = math.ceil(60 / HEADWAY_MAX_MIN)   # buses we leave on a trunk to hold the floor (=3)

## 1 — The efficiency grid

One row per `(line, direction, daytype, hour)`. `headway_min = 60 / trips_per_hour`. `demand_per_bus` is the key spare signal — **low = each running bus carries little**.

In [ ]:
def with_headway(df):
    return df.with_columns((60 / pl.col("trips_per_hour")).round(1).alias("headway_min"))

moment = (with_headway(eff.filter((pl.col("daytype") == DAYTYPE) & (pl.col("hour") == HOUR)))
          .sort("demand_per_bus"))
moment.select("line_id", "direction_id", "trips_per_hour", "headway_min",
              "served_demand", "demand_per_bus", "demand_per_km").head(15)

### Load-per-bus distribution across the network at this hour
The dashed line is the `DEMAND_PCTL` cutoff — lines left of it are the over-served / spare ones.

In [ ]:
m = eff.filter((pl.col("daytype") == DAYTYPE) & (pl.col("hour") == HOUR))
vals = m["demand_per_bus"].to_numpy()
cut = float(np.quantile(vals, DEMAND_PCTL))
plt.figure(figsize=(8, 3))
plt.hist(vals, bins=30, color="#4C78A8")
plt.axvline(cut, color="crimson", ls="--", label=f"{int(DEMAND_PCTL*100)}th pctl = {cut:.3f}")
plt.title(f"Load per bus across lines — {DAYTYPE} {HOUR:02d}:00")
plt.xlabel("demand_per_bus  (lower = more spare)"); plt.ylabel("# line-dirs")
plt.legend(); plt.tight_layout(); plt.show()

## 2 — The flex fleet + selector (Model B)

`recommend()` is the **selector**: it only ever picks from the *declared* flex fleet (`flex_eligible`, the Model-B trunks), keeps the ones that can actually spare a bus this hour, and ranks them by **`buses_freed`** — the donor that thins the most without breaking its headway floor goes first. This is the two-step the pitch needs — **designate first, then select** — and it mirrors `recommend()` in `pipeline/build_efficiency.py`.

In [ ]:
fleet = flex.filter(pl.col("flex_eligible")).select("line_id", "direction_id")

def recommend(daytype, hour, n=5):
    # declared flex buses that can spare a bus this hour, most spare first
    return (with_headway(
                eff.join(fleet, on=["line_id", "direction_id"], how="inner")
                   .filter((pl.col("daytype") == daytype) & (pl.col("hour") == hour)
                           & (pl.col("buses_freed") > 0) & (pl.col("served_demand") > 0)))
            .sort("buses_freed", descending=True)
            .select("line_id", "direction_id", "trips_per_hour", "headway_min",
                    "buses_freed", "served_demand", "demand_per_bus")
            .head(n))

print("Declared flex fleet:", fleet.height, "line-dirs")
recommend(DAYTYPE, HOUR)

### Where each line is cold — load-per-bus heatmap (line × hour)
Dark cells = spare. A line dark across the whole row is a permanent flex candidate; a line dark only off-peak can only flex then.

In [ ]:
pivot = (eff.filter(pl.col("daytype") == DAYTYPE)
         .pivot(values="demand_per_bus", index="line_id", on="hour", aggregate_function="mean")
         .sort("line_id"))
hours = sorted([c for c in pivot.columns if c != "line_id"], key=lambda x: int(x))
M = pivot.select(hours).to_numpy()
labels = pivot["line_id"].to_list()
plt.figure(figsize=(12, 9))
plt.imshow(M, aspect="auto", cmap="magma_r")
plt.colorbar(label="demand_per_bus  (dark = cold / spare)")
plt.yticks(range(len(labels)), labels, fontsize=7)
plt.xticks(range(len(hours)), hours, fontsize=8)
plt.xlabel("hour of day"); plt.ylabel("line"); plt.title(f"Load per bus — {DAYTYPE}")
plt.tight_layout(); plt.show()

## 3 — The thinning math (how a trunk becomes a donor)

This is the engine behind the designation above. Keep enough trips to hold `HEADWAY_MAX_MIN`; everything above that is spare:

```
trips_keep  = ceil(60 / HEADWAY_MAX_MIN)        # = KEEP_TRIPS
buses_freed = max(0, trips_per_hour - trips_keep)
```

A line is a **donor this hour** when `buses_freed > 0` *and* it runs through populated stops (`served_demand > 0` — confirms a real, demand-justified trunk, not a scheduling quirk). No low-demand cutoff: selection is pure frequency/redundancy. `demand_per_bus` is shown for context only.

In [ ]:
def thin_plan(daytype, hour, headway_max=HEADWAY_MAX_MIN):
    keep = math.ceil(60 / headway_max)
    m = eff.filter((pl.col("daytype") == daytype) & (pl.col("hour") == hour) & (pl.col("served_demand") > 0))
    return (with_headway(m)
        .with_columns(
            pl.lit(keep).alias("trips_keep"),
            (pl.col("trips_per_hour") - keep).clip(0).alias("buses_freed"),
            pl.lit(round(60 / keep, 1)).alias("headway_after"))
        .filter(pl.col("buses_freed") > 0)   # pure frequency: can it spare a bus and hold the floor?
        .sort("buses_freed", descending=True)
        .select("line_id", "direction_id", "trips_per_hour", "headway_min",
                "headway_after", "buses_freed", "demand_per_bus"))

thin_plan(DAYTYPE, HOUR)

### Line 1 across the day — your concrete example
How many buses would thinning Line 1 to the `HEADWAY_MAX_MIN` headway free, hour by hour?

In [ ]:
keep = math.ceil(60 / HEADWAY_MAX_MIN)
l1 = (with_headway(eff.filter((pl.col("line_id") == "1") & (pl.col("daytype") == DAYTYPE)
                              & (pl.col("direction_id") == 0)))
      .sort("hour")
      .with_columns(
          (pl.col("trips_per_hour") - keep).clip(0).alias("buses_freed_if_thinned"),
          pl.lit(round(60 / keep, 1)).alias("headway_after")))
l1.select("hour", "trips_per_hour", "headway_min", "headway_after",
          "buses_freed_if_thinned", "demand_per_bus")

## Network flex supply at the event hour
Total buses we could free by thinning over-served lines to the headway cap — capacity available to redeploy toward the event.

In [ ]:
plan = thin_plan(DAYTYPE, HOUR)
lab = [f"{r}/{d}" for r, d in zip(plan["line_id"], plan["direction_id"])]
plt.figure(figsize=(10, 4))
plt.bar(lab, plan["buses_freed"], color="#54A24B")
plt.title(f"Buses freeable by thinning to <= {HEADWAY_MAX_MIN}-min headway — {DAYTYPE} {HOUR:02d}:00")
plt.ylabel("buses/hour freed"); plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()
print("Total buses freeable this hour:", int(plan["buses_freed"].sum()))

## 4 — Determining the flex fleet: the knobs

`flex_candidates.parquet` flags a line-dir `flex_eligible` (Model B) when it can spare a bus — `buses_freed > 0` *and* `served_demand > 0` — for at least **`MIN_DONOR_HOURS`** hours of the day. Selection is **pure frequency/redundancy**: the trunks where a lot of buses already run. (The old Model-A bottom-25%-demand designation is gone — it picked near-empty peripheral lines whose one bus *is* the service.)

**Open questions to iterate on:**
- **Headway floor:** `HEADWAY_MAX_MIN=20` is a **guess** — RVV may publish a tighter service standard. The entire flex supply scales with it (`KEEP_TRIPS = ceil(60/floor)`); validate before quoting buses-freed numbers.
- **Donor-hours bar:** is `MIN_DONOR_HOURS=3` right? Raise it for a smaller, all-day-dependable fleet; lower it to admit peak-only trunks.
- **Whole buses:** `trips_per_hour` is a per-typical-day *rate*, so `buses_freed` is fractional (an expected spare). For the pitch, prefer donors with `peak_buses_freed ≥ 1` (frees a whole vehicle).
- **Normalized demand:** `demand_per_bus` is a relative proxy — context only here, can't promise "X seats free".
- **Proximity (not here yet):** a flex bus is only useful if it can *reach* the event. Next step: filter/rank `recommend()` by distance to the hotspot.

The map below shows the **declared fleet** (green) vs the rest (grey) — this is what riders would see badged as flex.

In [ ]:
import folium
fset = set(zip(fleet["line_id"], fleet["direction_id"]))
fmap = folium.Map(location=[49.013, 12.101], zoom_start=12, tiles="cartodbpositron")
for row in lines.iter_rows(named=True):
    g = row["geometry"]
    if not g:
        continue
    coords = json.loads(g)["coordinates"]
    latlon = [[c[1], c[0]] for c in coords]
    is_flex = (row["line_id"], row["direction_id"]) in fset
    folium.PolyLine(latlon,
                    color="#1b9e77" if is_flex else "#9aa0a6",
                    weight=4 if is_flex else 1.5,
                    opacity=0.95 if is_flex else 0.35,
                    tooltip=f'{row["line_id"]} dir{row["direction_id"]}' + (" — FLEX" if is_flex else "")
                    ).add_to(fmap)
fmap

## Next steps

- ~~Fold Model B into the designation~~ ✅ **done** — `flex_candidates` is now Model-B (frequency/redundancy); `build_efficiency.py` ships it.
- **Proximity-aware `recommend(hour, hotspot)`** — only surface flex buses that can reach the event corridor.
- **Wire to backend** — `GET /flex-fleet` (designation for map badges) + `recommend()` on event trigger.
- **Validate headway floor with RVV** — replace the 20-min guess with the real service standard.